# Week 3: Classification and Error Tradeoffs

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-03-classification-errors.ipynb)

**Estimated time:** 75–100 minutes  
**Deliverable:** a confusion matrix, two metrics, and an explanation of the more costly error.

### Learning goals

- Train a binary classifier on held-out data.
- Read true positives, false positives, and false negatives.
- Compare precision and recall when the decision threshold changes.


## 1. Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## 2. Create a synthetic support-ticket dataset

Class 1 represents tickets that need escalation. Class 0 represents tickets that the first support team can resolve. The data is synthetic and contains no real customer information.


In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.72, 0.28],
    class_sep=1.0,
    random_state=42,
)

print("rows and features:", X.shape)
print("escalation rate:", y.mean().round(3))


## 3. Split with stratification

Stratification keeps a similar class balance in the training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("training escalation rate:", y_train.mean().round(3))
print("test escalation rate:", y_test.mean().round(3))


## 4. Train logistic regression


In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]


## 5. Measure accuracy, precision, and recall


In [ ]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")


## 6. Read the confusion matrix


In [ ]:
matrix = confusion_matrix(y_test, predictions)
tn, fp, fn, tp = matrix.ravel()

print({"true_negative": tn, "false_positive": fp, "false_negative": fn, "true_positive": tp})

ConfusionMatrixDisplay(matrix, display_labels=["Standard", "Escalate"]).plot(cmap="Blues")
plt.title("Support-ticket classifier")
plt.show()


## 7. Change the decision threshold

A lower threshold flags more tickets for escalation. This often increases recall and also increases false positives.


In [ ]:
lower_threshold_predictions = (probabilities >= 0.35).astype(int)

lower_precision = precision_score(y_test, lower_threshold_predictions)
lower_recall = recall_score(y_test, lower_threshold_predictions)
lower_matrix = confusion_matrix(y_test, lower_threshold_predictions)

print(f"Precision at 0.35: {lower_precision:.3f}")
print(f"Recall at 0.35:    {lower_recall:.3f}")
print("Confusion matrix at 0.35:\n", lower_matrix)


## 8. Error-cost discussion

Answer in this cell:

1. What happens when a ticket that needs escalation receives a standard response?
2. What happens when a standard ticket is escalated unnecessarily?
3. Which error should matter more for this fictional support team, and why?


## 9. Your turn

Try thresholds of 0.25, 0.50, and 0.70. Record precision and recall for each. Choose one threshold and justify it using the error costs above.


## 10. Validation


In [ ]:
assert matrix.shape == (2, 2)
assert 0 <= precision <= 1
assert 0 <= recall <= 1
assert lower_recall >= recall
print("Validation passed. The lower threshold did not reduce recall.")


## Submission checklist

- Include the default confusion matrix.
- Report precision and recall for two thresholds.
- Identify the more costly error.
- Justify one threshold in plain language.
